# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umaimakhalid17/ML/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring.**

I'm picking this lane because the starter dataset is basically built for it: every row is a
published page with 90 days of search and engagement history, and the guide's own baseline
(`stale_visible_page`, `declining_with_demand`, `thin_visible_page`, `page_one_decay_risk`,
`low_ctr_visible_page`, `low_engagement_visible_page`) maps directly onto columns I can already
see in `content_refresh_anonymized.csv` (`days_since_last_update`, `impressions_90d`,
`word_count`, `avg_position`, `ctr`, `engagement_rate`). The other lanes are reasonable too —
signal analysis is a fallback if refresh scoring doesn't hold up, and clustering is tempting —
but refresh scoring is the one with an obvious human on the other end: a content editor with a
short list of pages and limited hours. That gives me a decision to design around from week one,
instead of a metric I have to invent a use for later. I'm not locking this in — I can revisit at
the end of Week 4 — but for a 7-week build, starting with the lane that already has a named
action beats starting with the one that's most interesting to explore.


In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("rows, columns:", df.shape)
print("distinct clients:", df["client_id"].nunique())
print()
print("trend_direction counts (this is where the starter label comes from):")
print(df["trend_direction"].value_counts())
print()
print("share of pages currently labeled 'down':",
      round((df["trend_direction"] == "down").mean() * 100, 1), "%")


rows, columns: (30000, 44)
distinct clients: 32

trend_direction counts (this is where the starter label comes from):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

share of pages currently labeled 'down': 54.2 %


## 2. The question: decision, action, cost of a wrong call

**Question:** Given a page's last 90 days of search and engagement data, which pages should a
content editor put at the top of this week's refresh queue?

**Decision it improves:** which handful of pages an editor with limited hours opens first —
right now that's picked by gut feel or a simple sort, not by a ranked, evidence-backed queue.

**Who acts, and what they do:** a content editor or SEO lead on the client-facing team. They take
the top of my ranked list, open each page, and decide to refresh the copy, rewrite the
title/meta, expand thin sections, or leave it — the recommendation is a starting point for their
review, not an automatic edit.

**Cost of a wrong call, in both directions:**
- *False positive* (I flag a page as worth reviewing and it wasn't): wasted editor time — maybe
  30–60 minutes per page checked that turns out fine. Annoying, not dangerous, but it erodes
  trust in the queue if it happens often.
- *False negative* (I miss a page that was actually declining with real demand behind it): a
  page that's already visible to search users keeps losing position or clicks unnoticed,
  potentially for months, because nobody was told to look at it. This is the costlier mistake —
  a page with real impressions behind it is worth protecting.

Because false negatives cost more than false positives here, I'll care about **recall among
high-impression pages** as much as precision@K when I get to validation, not just overall
accuracy.

**Why data or ML helps at all:** a single rule like "flag anything down 20%" already exists
(`trend_direction`) and it's not enough — the numbers below show it fires on over half the
dataset, which isn't a review queue, it's most of the inventory. The real judgment call is
weighing several signals at once (visibility, staleness, position, thinness, engagement) against
each other for a specific page, and that weighting is exactly the kind of messy, many-signal
pattern a plain if-statement handles badly but a scored/ranked model can learn and explain.


In [2]:
# If a naive rule flags every 'down' page, how big is that queue really?
down_mask = df["trend_direction"] == "down"
print("pages flagged by the naive rule ('trend_direction == down'):", down_mask.sum(),
      f"({down_mask.mean()*100:.1f}% of all 30,000 rows)")

# Editors can't review half the inventory every week -- how many of those flagged pages
# actually carry real search demand worth protecting?
down_with_demand = down_mask & (df["impressions_90d"] >= 100)
print("of those, pages with >=100 impressions in 90d (real demand):", down_with_demand.sum())

# And a second, unrelated rule from the lane guide's baseline -- how much do the candidate
# pools from different single rules overlap?
page_one_decay = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)
print("pages matching a totally different single rule (page_one_decay_risk):", page_one_decay.sum())
print("pages both flagged as 'down' AND page_one_decay_risk:", (down_mask & page_one_decay).sum())


pages flagged by the naive rule ('trend_direction == down'): 16262 (54.2% of all 30,000 rows)
of those, pages with >=100 impressions in 90d (real demand): 13152
pages matching a totally different single rule (page_one_decay_risk): 7076
pages both flagged as 'down' AND page_one_decay_risk: 3666


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter dataset that make this lane worth the next 7 weeks:


In [3]:
# 1. Scale: is there enough data, across enough clients, to learn something general?
print("1) Rows / clients:", df.shape[0], "rows across", df["client_id"].nunique(), "clients")
print("   -> enough clients to hold some out entirely for testing (client-grouped split).")
print()

# 2. Is the label distribution usable (not 99%/1%, not a coin flip that means nothing)?
down_rate = (df["trend_direction"] == "down").mean() * 100
print(f"2) 'down' label rate: {down_rate:.1f}% -- a workable base rate: common enough to learn")
print("   from, not so common (>90%) that flagging everything would already 'win'.")
print()

# 3. Does a cheap single-signal rule already separate "worth reviewing" pages, or is there
#    room for a learned score to add value on top of it?
thin_visible = (df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)
low_ctr_visible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)
print("3) Single reason codes barely overlap and each catches a different slice:")
print("   thin_visible_page:", thin_visible.sum(), "pages")
print("   low_ctr_visible_page:", low_ctr_visible.sum(), "pages")
print("   pages matching BOTH:", (thin_visible & low_ctr_visible).sum())
print("   -> no single rule covers the review-worthy pages; a score that blends signals")
print("      is doing real work here, not just relabeling one column.")


1) Rows / clients: 30000 rows across 32 clients
   -> enough clients to hold some out entirely for testing (client-grouped split).

2) 'down' label rate: 54.2% -- a workable base rate: common enough to learn
   from, not so common (>90%) that flagging everything would already 'win'.

3) Single reason codes barely overlap and each catches a different slice:
   thin_visible_page: 82 pages
   low_ctr_visible_page: 9759 pages
   pages matching BOTH: 18
   -> no single rule covers the review-worthy pages; a score that blends signals
      is doing real work here, not just relabeling one column.


## 4. Careful words: what I can and can't claim

**What I can claim, by the end of this project:**
- An **observed** association: pages sharing certain signal combinations (staleness + demand,
  or position + low CTR, etc.) were, in this 30,000-row anonymized slice, more likely to be
  labeled declining or thin or under-clicked.
- A **directional** signal: a page scoring high on my ranked queue is more likely, on average,
  to be worth an editor's attention than a randomly chosen page — measured against a defined,
  written-down label.
- A **decision-support** tool: a ranked list with reason codes an editor can inspect and
  override — never an automatic edit, never a guarantee.

**What I will never claim:**
- That refreshing a page **causes** it to recover — I have no experiment (no A/B test, no
  before/after design with a control group), only past observation. Causal language needs a
  causal design, which this data doesn't give me.
- Anything about **Google's actual ranking algorithm** — I only see search console-style
  outcomes (impressions, clicks, position), never Google's internal logic.
- That the starter label (`trend_direction == "down"`, a last-30d-vs-prev-30d bucket) is the
  *ideal* target — it's a **proxy label**, computed from the current window, not a true future
  outcome. A stronger version of this project (which I may build toward) would predict a
  **future** window (e.g., features from the prior 90 days → decline in the *next* 30 days)
  instead of relabeling the present. I'll be explicit in my write-up about which version I used.
- Any client-identifying detail — all IDs here are pseudonyms, used only for joining/grouping,
  never as features or as anything shown in output.


In [4]:
# Quick gut-check on the label-leakage trap the framing skill warns about:
# trend_direction (and trend_pct, which defines it) must never be used as a FEATURE,
# only as the thing we're trying to predict.
leak_prone_cols = [c for c in df.columns if c in ("trend_direction", "trend_pct")]
print("columns that define the label and must be excluded from features:", leak_prone_cols)

# And a reminder of scale: this starter CSV is a 30k-row slice of a ~79M-row warehouse --
# whatever I find here should be described as "on this slice", not "in general".
print("starter slice size: 30,000 rows  |  full warehouse fact table: 78,835,655 rows")


columns that define the label and must be excluded from features: ['trend_direction', 'trend_pct']
starter slice size: 30,000 rows  |  full warehouse fact table: 78,835,655 rows


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.